In [13]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, StratifiedGroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone


In [14]:
RANDOM_STATE = 42
CV = 5

In [15]:
df_train = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

In [16]:
df_summary = pd.DataFrame({'null': df_train.isna().sum(), 'unique': df_train.nunique(), 'type': df_train.dtypes, 'sample': df_train.iloc[0]})
df_summary

,null,unique,type,sample
PassengerId,0,891,int64,1
Survived,0,2,int64,0
Pclass,0,3,int64,3
Name,0,891,str,"Braund, Mr. Owen Harris"
Sex,0,2,str,male
Age,177,88,float64,22.0
SibSp,0,7,int64,1
Parch,0,7,int64,0
Ticket,0,681,str,A/5 21171
Fare,0,248,float64,7.25


In [17]:
for col in df_train.drop(columns=['PassengerId']).columns:
    print(col, df_train[col].unique()[0:7])

Survived [0 1]
Pclass [3 1 2]
Name <StringArray>
[                            'Braund, Mr. Owen Harris',
 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)',
                              'Heikkinen, Miss. Laina',
        'Futrelle, Mrs. Jacques Heath (Lily May Peel)',
                            'Allen, Mr. William Henry',
                                    'Moran, Mr. James',
                             'McCarthy, Mr. Timothy J']
Length: 7, dtype: str
Sex <StringArray>
['male', 'female']
Length: 2, dtype: str
Age [22. 38. 26. 35. nan 54.  2.]
SibSp [1 0 3 4 2 5 8]
Parch [0 1 2 5 3 4 6]
Ticket <StringArray>
[       'A/5 21171',         'PC 17599', 'STON/O2. 3101282',
           '113803',           '373450',           '330877',
            '17463']
Length: 7, dtype: str
Fare [ 7.25   71.2833  7.925  53.1     8.05    8.4583 51.8625]
Cabin <StringArray>
[nan, 'C85', 'C123', 'E46', 'G6', 'C103', 'D56']
Length: 7, dtype: str
Embarked <StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype

In [18]:
df_train['Cabin'].unique()

<StringArray>
[          nan,         'C85',        'C123',         'E46',          'G6',
        'C103',         'D56',          'A6', 'C23 C25 C27',         'B78',
 ...
        'B102',         'B69',         'E49',         'C47',         'D28',
         'E17',         'A24',         'C50',         'B42',        'C148']
Length: 148, dtype: str

In [19]:
COLUMN_TYPE = {
    'Age': 'num',
    'FamilySize': 'num',
    'Fare': 'num',
    'Sex': 'cat',
    'Embarked': 'cat',
    'Title': 'cat',
    'Pclass': 'ord',
    'GroupSize': 'ord',
    'Deck': 'none_cat'
}

FEATURE_GROUPS = [
    ('sex+class', ['Sex', 'Pclass']),
    ('+age/fare', ['Age', 'Fare']),
    ('+family', ['FamilySize']),
    ('+title', ['Title']),
    ('+group', ['GroupSize']),
    ('+deck', ['Deck']),
    ('+embarked', ['Embarked'])
]

In [20]:
def split_by_type(columns):
    result = {'num': [],'cat': [], 'none_cat': [], 'ord': []}
    for col in columns:
        t = COLUMN_TYPE[col]
        result[t].append(col)
    return result

In [21]:
counts = pd.concat([df_train['Ticket'], df_test['Ticket']]).value_counts()
def add_features(df):
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['GroupSize'] = df['Ticket'].map(counts)
    df['Title'] = df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()
    df['Deck'] = df['Cabin'].str[0]
    return df

In [22]:
df_train = add_features(df_train)
df_test = add_features(df_test)

In [23]:
df_train['Title'].value_counts()
COMMON_TITLES = ['Mr', 'Miss', 'Mrs', 'Master']

def group_rare_title(df):
    df['Title'] = df['Title'].where(df['Title'].isin(COMMON_TITLES), 'other')
    return df
df_train = group_rare_title(df_train)
df_test = group_rare_title(df_test)

age_by_title = pd.concat([df_train, df_test]).groupby('Title')['Age'].median()

def fill_age_by_title(df):
    df['Age'] = df['Age'].fillna(df['Title'].map(age_by_title))
    return df

df_train = fill_age_by_title(df_train)
df_test = fill_age_by_title(df_test)

In [24]:
X_train = df_train.drop(columns=['PassengerId', 'Survived'])
y_train = df_train['Survived']
X_test = df_test.drop(columns=['PassengerId'])
test_id = df_test['PassengerId']
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
groups = df_train['Ticket']

In [25]:

def get_dummy_prediction(X, y):
    dummy = DummyClassifier(strategy='constant', constant=0, random_state=42)
    score = -cross_val_score(estimator=dummy, X=X, y=y, scoring='accuracy', groups=groups, cv=cv)
    return score

def build_preprocessor(num_cols, cat_cols, none_cat_cols, ord_cols):
    num_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
    cat_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
    none_cat_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='constant', fill_value='None')), ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))])
    ord_pipeline = Pipeline(steps=[('impute', SimpleImputer(strategy='most_frequent'))])
    preprocessor = ColumnTransformer(transformers=[('num', num_pipeline, num_cols), ('cat', cat_pipeline, cat_cols), ('none_cat', none_cat_pipeline, none_cat_cols), ('ord', ord_pipeline, ord_cols)])
    return preprocessor

def build_pipeline(preprocessor, model):
    full_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    return full_pipeline

def score_columns(chosen, model): 
    parts = split_by_type(chosen)
    preprocessor = build_preprocessor(num_cols=parts['num'], cat_cols=parts['cat'], none_cat_cols=parts['none_cat'], ord_cols=parts['ord'])
    pipeline = build_pipeline(preprocessor=preprocessor,  model=clone(model))
    scores = cross_val_score(pipeline, X=X_train, y=y_train, cv=cv, groups=groups, scoring='accuracy')
    return scores.mean()    

models = {
    'xgboost': XGBClassifier(random_state=RANDOM_STATE),
    'catboost': CatBoostClassifier(random_state=RANDOM_STATE, allow_writing_files=False, verbose=0),
    'random_forest': RandomForestClassifier(random_state=RANDOM_STATE),
    'logistic': LogisticRegression(random_state=RANDOM_STATE, max_iter=10000)
}

param_distributions = {
    'xgboost': {'model__max_depth': [2,3,4,5], 'model__learning_rate': [0.01, 0.03, 0.1], 'model__n_estimators': [100,300,600], 'model__subsample': [0.7, 0.85, 1.0], 'model__colsample_bytree': [0.6, 0.8, 1.0], 'model__min_child_weight': [1,3,5]}
    ,'catboost': {'model__max_depth': [3,4,5,6], 'model__learning_rate': [0.03, 0.05, 0.1], 'model__iterations': [300,500,1000], 'model__l2_leaf_reg': [1,3,5,10]},
        'random_forest': {'model__max_depth': [3,4,5,6,8, None], 'model__min_samples_leaf': [1,2,4,8,12], 'model__min_samples_split': [2,5,10,20], 'model__max_features': ['sqrt', 'log2', 0.5], 'model__max_samples': [0.6,0.8,None]},
    'logistic': {'model__C': [0.01, 0.1, 1, 10]}
}

family = {
    'xgboost': 'trees',
    'catboost': 'trees',
    'random_forest': 'trees',
    'logistic': 'linear'
}
def evaluate_model(model_name, model):
    pipeline = build_pipeline(preprocessor=preprocessor, model=model)
    accuracy = cross_val_score(estimator=pipeline, X=X_train, y=y_train, groups=groups, cv=cv, scoring='accuracy')
    return {
        'model': model_name,
        'accuracy': accuracy
    }
    
def select_best_result(results):
    if results.empty:
        raise ValueError('Some error happened')
    else:
        best_result = results.loc[results['validation_accuracy'].idxmax()]
        return best_result

def tune_model(model_name, model, param_distributions, n_iter=30):
    pipeline = build_pipeline(preprocessor=preprocessor, model=model)
    search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_distributions, n_iter=n_iter, scoring='accuracy', n_jobs=-1, cv=cv, return_train_score=True, refit=False, random_state=RANDOM_STATE, error_score='raise')
    search.fit(X_train, y_train, groups=groups)
    results = pd.DataFrame(search.cv_results_)
    results['model'] = model_name
    results['validation_accuracy'] = results['mean_test_score']
    results['train_accuracy'] = results['mean_train_score']
    best_result = select_best_result(results=results)
    return search, results, best_result


    

In [26]:
rows, used = [], []

for name, new_cols in FEATURE_GROUPS:
    used = used + new_cols
    row = {'step': name, 'n_features': len(used)}
    for model_name, model in models.items():
        row[model_name] = score_columns(chosen=used, model=model)
    rows.append(row)
ablation_results = pd.DataFrame(rows)
ablation_results

,step,n_features,xgboost,catboost,random_forest,logistic
0,sex+class,2,0.774406,0.774406,0.774406,0.786784
1,+age/fare,4,0.802502,0.828276,0.804706,0.781129
2,+family,5,0.803582,0.827190,0.811454,0.784512
3,+title,6,0.818176,0.830536,0.825997,0.828333
4,+group,7,0.810311,0.831678,0.821509,0.822696
5,+deck,8,0.808114,0.828289,0.819205,0.822645
6,+embarked,9,0.811498,0.826067,0.801297,0.827165


In [27]:
FINAL = ['Sex', 'Pclass', 'Age', 'Fare', 'FamilySize', 'Title']
final_features = split_by_type(FINAL)
preprocessor = build_preprocessor(num_cols=final_features['num'], cat_cols=final_features['cat'], none_cat_cols=final_features['none_cat'], ord_cols=final_features['ord'])

In [28]:
searches = {}
tuning_results = []
best_results = []
for model_name, model in models.items():
    search, results, best_result = tune_model(model_name=model_name, model=model, param_distributions=param_distributions[model_name])
    searches[model_name] = search
    tuning_results.append(results)
    best_results.append(best_result)
    
all_tuning_results = pd.concat(tuning_results, ignore_index=True)
best_results_df = pd.DataFrame(best_results).sort_values(by='validation_accuracy', ascending=False)


c:\Users\yazid\anaconda3\envs\ds_env\Lib\site-packages\sklearn\model_selection\_search.py:327: UserWarning: The total space of parameters 4 is smaller than n_iter=30. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [29]:
best_results_df['family'] = best_results_df['model'].map(family)
best_model_per_family = best_results_df.loc[best_results_df.groupby('family')['validation_accuracy'].idxmax()]
best_model_per_family['model']


2     logistic
19     xgboost
Name: model, dtype: str

In [31]:


def make_submission(best_models):
    predictions_proba = []
    for _, results in best_models.iterrows():
        model = clone(models[results['model']])
        pipeline = build_pipeline(preprocessor=preprocessor, model=model)
        pipeline.set_params(**results['params'])
        pipeline.fit(X_train, y_train)
        prediction_proba = pipeline.predict_proba(X_test)[:,1]
        predictions_proba.append(prediction_proba)
    avg_proba = np.mean(predictions_proba, axis=0)
    ensemble_prediction = (avg_proba > 0.5).astype(int)
    submission = pd.DataFrame({'PassengerId': test_id, 'Survived': ensemble_prediction})
    submission.to_csv('../submission5.csv', index=False)
    

In [32]:
make_submission(best_models=best_model_per_family)